In [1]:
# ==========================================================
# HEARTGUARD - MASTER DATASET MODEL TRAINING
# ==========================================================

import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# ----------------------------------------------------------
# PATHS
# ----------------------------------------------------------

DATA_PATH = os.path.join("..", "data", "final")
MODEL_PATH = os.path.join("..", "backend", "models")

# ----------------------------------------------------------
# LOAD MASTER DATASET
# ----------------------------------------------------------

df = pd.read_csv(os.path.join(DATA_PATH, "heartguard_master_dataset_v1.csv"))

print("Original Shape:", df.shape)

# ----------------------------------------------------------
# 1️⃣ DROP HIGHLY MISSING COLUMNS (>60%)
# ----------------------------------------------------------

missing_percent = df.isnull().mean() * 100

print("\nMissing %:\n", missing_percent)

cols_to_drop = missing_percent[missing_percent > 60].index.tolist()

print("\nDropping columns:", cols_to_drop)

df = df.drop(columns=cols_to_drop)

# ----------------------------------------------------------
# 2️⃣ HANDLE REMAINING MISSING VALUES
# ----------------------------------------------------------

X = df.drop(columns=["target"])
y = df["target"]

imputer = SimpleImputer(strategy="median")
X = imputer.fit_transform(X)

# ----------------------------------------------------------
# 3️⃣ FEATURE SCALING
# ----------------------------------------------------------

scaler = StandardScaler()
X = scaler.fit_transform(X)

# ----------------------------------------------------------
# 4️⃣ TRAIN TEST SPLIT
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------------------------------
# 5️⃣ MODEL TRAINING
# ----------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)

model.fit(X_train, y_train)

# ----------------------------------------------------------
# 6️⃣ EVALUATION
# ----------------------------------------------------------

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)

print("\n📊 MASTER MODEL PERFORMANCE")
print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}")

# ----------------------------------------------------------
# 7️⃣ SAVE MODEL
# ----------------------------------------------------------

joblib.dump(model, os.path.join(MODEL_PATH, "master_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_PATH, "master_scaler.pkl"))
joblib.dump(imputer, os.path.join(MODEL_PATH, "master_imputer.pkl"))

print("\n✅ Master model saved successfully!")

Original Shape: (312774, 12)

Missing %:
 age                   0.000000
sex                   0.000000
bmi                   0.101671
cholesterol           0.111582
systolic_bp          73.462948
diastolic_bp         73.462948
glucose              76.388702
diabetes             22.380377
smoking               0.000000
hypertension         25.182080
physical_activity     1.450568
target                0.000000
dtype: float64

Dropping columns: ['systolic_bp', 'diastolic_bp', 'glucose']

📊 MASTER MODEL PERFORMANCE
Accuracy: 0.8146
ROC-AUC : 0.8027

✅ Master model saved successfully!
